In [1]:
import warnings
warnings.filterwarnings("ignore")

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local-cluster[3,2,2048]")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(sc.getConf().get("spark.driver.memory", "NOT SET"))

2g


In [2]:
my_schema = """
VendorID INT,
tpep_pickup_datetime TIMESTAMP,
tpep_dropoff_datetime TIMESTAMP,
passenger_count DOUBLE,
trip_distance DOUBLE,
RatecodeID DOUBLE,
store_and_fwd_flag STRING,
PULocationID INT,
DOLocationID INT,
payment_type INT,
fare_amount DOUBLE,
extra DOUBLE,
mta_tax DOUBLE,
tip_amount DOUBLE,
tolls_amount DOUBLE,
improvement_surcharge DOUBLE,
total_amount DOUBLE,
congestion_surcharge DOUBLE,
Airport_fee DOUBLE,
cbd_congestion_fee DOUBLE
"""

In [3]:
df = spark.read.format('csv').option('header', 'true').schema(my_schema).load('yellow_tripdata_combined.csv')

In [4]:
from pyspark.sql.functions import *

In [5]:
filter_df = df.filter(col('passenger_count') == 1)
spark.sparkContext.setJobDescription('Filter data ---> Write')
filter_df.write.format('noop').mode('overwrite').save()

In [6]:
select_df = df.select('passenger_count', 'fare_amount', 'tip_amount')
spark.sparkContext.setJobDescription('Select columns ---> Write')
select_df.write.format('noop').mode('overwrite').save()

In [7]:
transform_df = df.withColumn(
    "diff_minutes",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60
)
spark.sparkContext.setJobDescription('Add new column ---> Write')
transform_df.write.format('noop').mode('overwrite').save()

In [8]:
df.cache()
spark.sparkContext.setJobDescription('Cache df')
transform_df.write.format('noop').mode('overwrite').save()

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp, tpep_dropoff_datetime: timestamp, passenger_count: double, trip_distance: double, RatecodeID: double, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: int, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double]

In [9]:
filter_df = df.filter(col('passenger_count') == 1)
spark.sparkContext.setJobDescription('Filter data ---> Write (after caching)')
filter_df.write.format('noop').mode('overwrite').save()

In [10]:
select_df = df.select('passenger_count', 'fare_amount', 'tip_amount')
spark.sparkContext.setJobDescription('Select columns ---> Write (after caching)')
select_df.write.format('noop').mode('overwrite').save()

In [11]:
transform_df = df.withColumn(
    "diff_minutes",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60
)
spark.sparkContext.setJobDescription('Add new column ---> Write (after caching))')
transform_df.write.format('noop').mode('overwrite').save()